# Project 2 — Healthcare Claims Data Quality & Anomaly Detection

**Tools:** SQLite / SQL • Python / Pandas / NumPy • Power BI

This notebook uses the supplied synthetic healthcare claims dataset. The goal is to validate the claims, understand utilization and financial patterns, identify unusual operational patterns, and prepare Power BI-ready reporting.

> **Important:** An anomaly flag is a screening signal for review. It is not proof of fraud or misconduct.

## Roadmap

`Raw data → SQL quality checks → SQL business analysis → Python enrichment → anomaly detection → evaluation → Power BI`

The notebook uses small, readable cells so that every step can be explained line-by-line in an interview.

In [ ]:
from pathlib import Path
import sqlite3

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

In [ ]:
PROJECT_ROOT = Path("/mnt/data/innovaccer_project2_claims_anomaly_project")
DATABASE_PATH = PROJECT_ROOT / "01_data_raw" / "innovaccer_claims.db"
OUTPUT_PATH = PROJECT_ROOT / "05_outputs"

print(DATABASE_PATH)

## 1. Connect to SQLite and inspect the tables

In [ ]:
connection = sqlite3.connect(DATABASE_PATH)

tables = pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name
    """,
    connection,
)

tables

In [ ]:
patients = pd.read_sql_query("SELECT * FROM patients", connection)
providers = pd.read_sql_query("SELECT * FROM providers", connection)
facilities = pd.read_sql_query("SELECT * FROM facilities", connection)
claims = pd.read_sql_query("SELECT * FROM claims", connection)
claim_lines = pd.read_sql_query("SELECT * FROM claim_lines", connection)
diagnosis = pd.read_sql_query("SELECT * FROM diagnosis_catalog", connection)
procedure = pd.read_sql_query("SELECT * FROM procedure_catalog", connection)
truth = pd.read_csv(PROJECT_ROOT / "01_data_raw" / "anomaly_ground_truth.csv")

In [ ]:
pd.DataFrame({
    "Table": [
        "patients", "providers", "facilities", "claims",
        "claim_lines", "diagnosis_catalog", "procedure_catalog"
    ],
    "Rows": [
        len(patients), len(providers), len(facilities), len(claims),
        len(claim_lines), len(diagnosis), len(procedure)
    ]
})

## 2. Inspect and type the claims data

In [ ]:
claims.head(10)

In [ ]:
claims.info()

In [ ]:
claims["Service_Date"] = pd.to_datetime(claims["Service_Date"], errors="coerce")
claims["Submission_Date"] = pd.to_datetime(claims["Submission_Date"], errors="coerce")

In [ ]:
financial_columns = [
    "Claim_Amount",
    "Allowed_Amount",
    "Paid_Amount",
    "Member_Responsibility",
]

for column in financial_columns:
    claims[column] = pd.to_numeric(
        claims[column],
        errors="coerce"
    )

## 3. Data-quality checks

In [ ]:
duplicate_claim_rows = claims["Claim_ID"].duplicated(keep=False).sum()
missing_provider = claims["Provider_ID"].fillna("").eq("").sum()
missing_line_diagnosis = claim_lines["Diagnosis_Code"].fillna("").eq("").sum()
paid_gt_allowed = (claims["Paid_Amount"] > claims["Allowed_Amount"]).sum()
submission_before_service = (claims["Submission_Date"] < claims["Service_Date"]).sum()

print("Duplicate claim rows:", duplicate_claim_rows)
print("Missing provider IDs:", missing_provider)
print("Missing line diagnoses:", missing_line_diagnosis)
print("Paid > allowed:", paid_gt_allowed)
print("Submission before service:", submission_before_service)

## 4. Build the enriched fact table

In [ ]:
fact = (
    claims
    .merge(
        patients[["Patient_ID", "Age", "Gender", "Region"]],
        on="Patient_ID",
        how="left"
    )
    .merge(
        providers[["Provider_ID", "Provider_Display_Name", "Specialty"]],
        on="Provider_ID",
        how="left"
    )
    .merge(
        facilities[["Facility_ID", "Facility_Name", "Facility_Type", "Facility_Size"]],
        on="Facility_ID",
        how="left"
    )
    .merge(
        diagnosis[["Diagnosis_Code", "Diagnosis_Name", "Clinical_Group"]],
        on="Diagnosis_Code",
        how="left"
    )
    .merge(
        procedure[["Procedure_Code", "Procedure_Name", "Procedure_Group"]],
        left_on="Primary_Procedure_Code",
        right_on="Procedure_Code",
        how="left"
    )
)

fact.shape

In [ ]:
fact["Submission_Lag_Days"] = (
    fact["Submission_Date"] - fact["Service_Date"]
).dt.days

fact["Month"] = (
    fact["Service_Date"]
    .dt.to_period("M")
    .astype(str)
)

fact["Week_Start"] = (
    fact["Service_Date"]
    - pd.to_timedelta(
        fact["Service_Date"].dt.weekday,
        unit="D"
    )
).dt.date.astype(str)

fact["Paid_Rate"] = (
    fact["Paid_Amount"]
    / fact["Allowed_Amount"].replace(0, np.nan)
)

## 5. Descriptive analysis: payer performance

In [ ]:
payer_summary = (
    fact
    .groupby("Payer", as_index=False)
    .agg(
        Claims=("Claim_ID", "count"),
        Patients=("Patient_ID", "nunique"),
        Billed=("Claim_Amount", "sum"),
        Allowed=("Allowed_Amount", "sum"),
        Paid=("Paid_Amount", "sum"),
        Denied_Rejected=(
            "Claim_Status",
            lambda values: values.isin(["Denied", "Rejected"]).sum()
        ),
    )
)

payer_summary["Payment_Rate"] = payer_summary["Paid"] / payer_summary["Allowed"]
payer_summary["Denial_Rejection_Rate"] = payer_summary["Denied_Rejected"] / payer_summary["Claims"]

payer_summary.sort_values("Claims", ascending=False)

## 6. Descriptive analysis: specialty and provider volume

In [ ]:
specialty_summary = (
    fact
    .groupby("Specialty", as_index=False)
    .agg(
        Claims=("Claim_ID", "count"),
        Patients=("Patient_ID", "nunique"),
        Avg_Claim=("Claim_Amount", "mean"),
        Billed=("Claim_Amount", "sum"),
        Paid=("Paid_Amount", "sum"),
    )
    .sort_values("Billed", ascending=False)
)

specialty_summary

In [ ]:
provider_summary = (
    fact[fact["Provider_ID"].fillna("").ne("")]
    .groupby(["Provider_ID", "Provider_Display_Name", "Specialty"], as_index=False)
    .agg(
        Claims=("Claim_ID", "count"),
        Patients=("Patient_ID", "nunique"),
        Avg_Claim=("Claim_Amount", "mean"),
        Billed=("Claim_Amount", "sum"),
        Paid=("Paid_Amount", "sum"),
    )
    .sort_values("Claims", ascending=False)
)

provider_summary.head(20)

## 7. Claim-value distribution and threshold selection

In [ ]:
fact["Claim_Amount"].describe(
    percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
)

In [ ]:
p99 = fact["Claim_Amount"].quantile(0.99)

q1 = fact["Claim_Amount"].quantile(0.25)
q3 = fact["Claim_Amount"].quantile(0.75)
iqr_upper = q3 + 1.5 * (q3 - q1)

print("P99 threshold:", round(p99, 2))
print("IQR upper fence:", round(iqr_upper, 2))

## 8. Anomaly rule: high-value claims

In [ ]:
fact["High_Value_P99"] = (
    fact["Claim_Amount"] >= p99
).astype(int)

high_value_claims = fact[
    fact["High_Value_P99"] == 1
].sort_values("Claim_Amount", ascending=False)

high_value_claims[
    [
        "Claim_ID",
        "Provider_ID",
        "Specialty",
        "Payer",
        "Procedure_Name",
        "Claim_Amount",
    ]
].head(20)

## 9. Anomaly rule: provider weekly volume spikes

In [ ]:
provider_weekly = (
    fact[fact["Provider_ID"].fillna("").ne("")]
    .groupby(["Provider_ID", "Week_Start"])
    .size()
    .reset_index(name="Weekly_Claims")
)

provider_baseline = (
    provider_weekly
    .groupby("Provider_ID")["Weekly_Claims"]
    .agg(["mean", "std"])
    .reset_index()
    .rename(columns={
        "mean": "Mean_Weekly_Claims",
        "std": "Std_Weekly_Claims"
    })
)

provider_weekly = provider_weekly.merge(
    provider_baseline,
    on="Provider_ID",
    how="left"
)

provider_weekly["Volume_Z"] = (
    provider_weekly["Weekly_Claims"]
    - provider_weekly["Mean_Weekly_Claims"]
) / provider_weekly["Std_Weekly_Claims"].replace(0, np.nan)

volume_spikes = provider_weekly[
    (provider_weekly["Volume_Z"] >= 3)
    & (provider_weekly["Weekly_Claims"] >= 8)
]

volume_spikes.sort_values("Volume_Z", ascending=False).head(20)

## 10. Anomaly rule: financial exceptions

In [ ]:
financial_exceptions = fact[
    fact["Paid_Amount"] > fact["Allowed_Amount"]
]

financial_exceptions[
    ["Claim_ID", "Payer", "Claim_Amount", "Allowed_Amount", "Paid_Amount"]
].head(20)

## 11. Combine operational screening rules

In [ ]:
fact["Missing_Provider"] = (
    fact["Provider_ID"].fillna("").eq("")
).astype(int)

spike_keys = set(
    zip(
        volume_spikes["Provider_ID"],
        volume_spikes["Week_Start"]
    )
)

fact["Provider_Week_Spike"] = [
    int((provider, week) in spike_keys)
    for provider, week
    in zip(fact["Provider_ID"], fact["Week_Start"])
]

fact["Financial_Exception"] = (
    fact["Paid_Amount"] > fact["Allowed_Amount"]
).astype(int)

fact["Anomaly_Flag"] = (
    (fact["High_Value_P99"] == 1)
    | (fact["Financial_Exception"] == 1)
    | (fact["Missing_Provider"] == 1)
    | (fact["Provider_Week_Spike"] == 1)
).astype(int)

fact["Anomaly_Flag"].value_counts()

## 12. Evaluate against the separate synthetic ground truth

In [ ]:
evaluation = (
    fact[["Claim_ID", "Anomaly_Flag"]]
    .drop_duplicates("Claim_ID")
    .merge(
        truth[["Claim_ID", "Ground_Truth_Flag"]].drop_duplicates("Claim_ID"),
        on="Claim_ID",
        how="left"
    )
)

evaluation["Ground_Truth_Flag"] = (
    evaluation["Ground_Truth_Flag"]
    .fillna(0)
    .astype(int)
)

In [ ]:
true_positive = ((evaluation["Anomaly_Flag"] == 1) & (evaluation["Ground_Truth_Flag"] == 1)).sum()
false_positive = ((evaluation["Anomaly_Flag"] == 1) & (evaluation["Ground_Truth_Flag"] == 0)).sum()
false_negative = ((evaluation["Anomaly_Flag"] == 0) & (evaluation["Ground_Truth_Flag"] == 1)).sum()

precision = true_positive / (true_positive + false_positive) if true_positive + false_positive else 0
recall = true_positive / (true_positive + false_negative) if true_positive + false_negative else 0
f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0

pd.DataFrame({
    "Metric": ["True Positives", "False Positives", "False Negatives", "Precision", "Recall", "F1"],
    "Value": [true_positive, false_positive, false_negative, precision, recall, f1],
})

## 13. Export Power BI-ready files

In [ ]:
OUTPUT_PATH.mkdir(exist_ok=True)

fact.to_csv(OUTPUT_PATH / "fact_claims_enriched.csv", index=False)
payer_summary.to_csv(OUTPUT_PATH / "payer_summary.csv", index=False)
provider_summary.to_csv(OUTPUT_PATH / "provider_summary.csv", index=False)

print("Export complete:", OUTPUT_PATH)

## Interview takeaway

> I treated the project as an operational healthcare analytics workflow. I validated the raw claims in SQL, enriched them with patient/provider/facility/diagnosis/procedure dimensions, used Python for distribution analysis and explainable anomaly rules, and prepared Power BI-ready reporting. Each anomaly is a review signal rather than proof of fraud.